In [10]:
import json

from torch.fx.experimental.symbolic_shapes import create_contiguous

with open("/Users/nad/mobiraph/data/n13_repbase_processed/hierarchy_sequences_02_ltr_correction_with_classes.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(type(metadata))

<class 'dict'>


In [11]:
general_dict = {}

In [12]:
for class_name, class_info in metadata.items():
    sequences = class_info["sequences"]
    for sequence in sequences:
        general_dict[sequence] = {"class": class_name}

In [17]:
for class_name, class_info in metadata.items():
    if class_name == 'Class II (DNA transposons)':
        sequences = class_info["sequences"]
        for sequence in sequences:
            general_dict[sequence]["order"] = "DNA transposon"

In [19]:
general_dict['MARINER62_CB']

{'class': 'Class II (DNA transposons)',
 'superfamily': 'Mariner/Tc1',
 'order': 'DNA transposon'}

In [14]:
for supfam_name, supfam_info in metadata['Class II (DNA transposons)']["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [23]:
for order_name, order_info in metadata['Class I (Retrotransposons)']["subs"].items():
    sequences = order_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["order"] = order_name

In [24]:
general_dict['LTR4_CR-LTR']

{'class': 'Class I (Retrotransposons)', 'order': 'LTR Retrotransposon'}

In [25]:
for supfam_name, supfam_info in metadata['Class I (Retrotransposons)']["subs"]["LTR Retrotransposon"]["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [29]:
general_dict['GYPSY1-LTR_CB']

{'class': 'Class I (Retrotransposons)',
 'order': 'LTR Retrotransposon',
 'superfamily': 'Gypsy'}

In [30]:
for supfam_name, supfam_info in metadata['Class I (Retrotransposons)']["subs"]["Non-LTR Retrotransposon"]["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [32]:
general_dict['SINEX-1_CR']

{'class': 'Class I (Retrotransposons)',
 'order': 'Non-LTR Retrotransposon',
 'superfamily': 'SINE'}

In [36]:
with open("/Users/nad/mobiraph/data/n13_repbase_processed/metadata_03.json", "w", encoding="utf-8") as f:
    json.dump(general_dict, f, ensure_ascii=False, indent=4)

In [54]:
sv_plants_category = {}

with open("/Users/nad/mobiraph/data/plant_sv_fam_orf_on_repbase_best.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_plants_category[parts[0]] = general_dict[parts[1]]

In [58]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/sv_plants_category.json", "w", encoding="utf-8") as f:
    json.dump(sv_plants_category, f, ensure_ascii=False, indent=4)

In [55]:
len(sv_plants_category)

11914

In [56]:
sv_insects_category = {}

with open("/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_insects_category[parts[0]] = general_dict[parts[1]]

In [59]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/sv_insects_category.json", "w", encoding="utf-8") as f:
    json.dump(sv_insects_category, f, ensure_ascii=False, indent=4)

In [60]:
len(sv_insects_category)

7773

In [92]:
def create_hierarchy_dict(g_dict):
    hierarchy_sequences_sv = {}
    # class level
    hierarchy_sequences_sv['Class I (Retrotransposons)'] = {'sequences': [], 'subs': {}}
    hierarchy_sequences_sv['Class II (DNA transposons)'] = {'sequences': [], 'subs': {}}
    for name, info in g_dict.items():
        hierarchy_sequences_sv[info['class']]['sequences'].append(name)
    # order level
    hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['LTR Retrotransposon'] = {'sequences': [], 'subs': {}}
    hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon'] = {'sequences': [], 'subs': {}}
    for name, info in g_dict.items():
        if not info['order']:
            continue
        if info['class'] == 'Class I (Retrotransposons)':
            hierarchy_sequences_sv['Class I (Retrotransposons)']['subs'][info['order']]['sequences'].append(name)
    # superfamily level
    for superfamily in metadata['Class I (Retrotransposons)']['subs']['LTR Retrotransposon']['subs'].keys():
        hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['LTR Retrotransposon']['subs'][superfamily] = {'sequences': [], 'subs': {}}
    for superfamily in metadata['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon']['subs'].keys():
        hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon']['subs'][superfamily] = {'sequences': [], 'subs': {}}
    for superfamily in metadata['Class II (DNA transposons)']['subs'].keys():
        hierarchy_sequences_sv['Class II (DNA transposons)']['subs'][superfamily] = {'sequences': [], 'subs': {}}

    for name, info in g_dict.items():
        if 'superfamily' not in info:
            continue
        if info['class'] == 'Class I (Retrotransposons)':
            hierarchy_sequences_sv['Class I (Retrotransposons)']['subs'][info['order']]['subs'][info['superfamily']]['sequences'].append(name)
        if info['class'] == 'Class II (DNA transposons)':
            hierarchy_sequences_sv['Class II (DNA transposons)']['subs'][info['superfamily']]['sequences'].append(name)
    return hierarchy_sequences_sv


In [93]:
hierarchy_sequences_sv_plants = create_hierarchy_dict(sv_plants_category)
hierarchy_sequences_sv_insects = create_hierarchy_dict(sv_insects_category)

In [95]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_plants.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_sv_plants, f, ensure_ascii=False, indent=4)
with open("/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_insects.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_sv_insects, f, ensure_ascii=False, indent=4)

In [99]:
sv_plants_category["plant_phaseolus_vulgaris|SVgr_3_id_085775|7909"]

{'class': 'Class I (Retrotransposons)',
 'order': 'Non-LTR Retrotransposon',
 'superfamily': 'L1'}

In [106]:
files = []
for i in range(10):
    files.append(f"/Users/nad/NeuralTE/plants_output/domain/{i}.out")

with open("/Users/nad/NeuralTE/plants_output/domain/all.out", "w", encoding="utf-8") as outfile:
    for fname in files:
        with open(fname, "r", encoding="utf-8") as infile:
            outfile.write(infile.read())

In [107]:
neuralte_results = {}

with open("/Users/nad/NeuralTE/plants_output/domain/all.out") as f:
    for line in f:
        parts = line.strip().split("\t")

        name = parts[0]
        type_full = parts[1]

        type_clean = type_full.split("#")[-1]

        neuralte_results[name] = type_clean

print(neuralte_results["plant_oryza_meridionalis|SVgr_11_id_16072|27066"])

LTR/Gypsy


In [115]:
len(set(neuralte_results.values()))

24

In [110]:
all_count = 0
true_count = 0

def has_common_substring(s1, s2, min_len=2):
    for i in range(len(s1) - min_len + 1):
        sub = s1[i:i+min_len]
        if sub in s2:
            return True
    return False


for name in neuralte_results.keys():
    if name not in sv_plants_category:
        continue
    all_count += 1
    # if 'superfamily' not in sv_plants_category[name].keys():
    #     if has_common_substring(neuralte_results[name],
    #                         sv_plants_category[name]['order']):
    #         true_count += 1
    #     else:
    #         print(neuralte_results[name], sv_plants_category[name]['order'])
    if 'superfamily' not in sv_plants_category[name].keys():
        continue
    else:
        if has_common_substring(neuralte_results[name],
                            sv_plants_category[name]['superfamily']):
            true_count += 1
        else:
            print(neuralte_results[name], sv_plants_category[name]['order'])

DNA/hAT-Ac LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LINE/L1 DNA transposon
LTR/Copia LTR Retrotransposon
LTR/Copia DNA transposon
LTR/Gypsy LTR Retrotransposon
LINE/L1 Non-LTR Retrotransposon
DNA/MULE-MuDR DNA transposon
DNA/MULE-MuDR LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
LTR/Gypsy DNA transposon
LTR/Caulimovirus DNA transposon
LINE/L1 DNA transposon
LTR/Gypsy LTR Retrotransposon
LTR/Gypsy Non-LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
DNA/CMC-EnSpm DNA transposon
DNA/CMC-EnSpm LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
DNA/PIF-Harbinger, LTR Retrotransposon
DNA/PIF-Harbinger LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Gypsy Non-LTR Retrotransposon
DNA/Ginger-1 LTR Retrotransposon
LTR/Copia DNA transpos

In [111]:
true_count / all_count

0.9863599893019523

In [112]:
true_count, all_count

(11064, 11217)

# Предсказания на кусочках

In [116]:
import pandas as pd


def load_name_to_class(csv_path: str) -> dict[str, str]:
    df = pd.read_csv(csv_path)

    if "name" not in df.columns or "y_pred" not in df.columns:
        raise ValueError("Ожидаются колонки 'name' и 'y_pred'")

    return dict(zip(df["name"], df["y_pred"]))


# пример
name_to_pred_class = load_name_to_class(
    f"/Users/nad/mobiraph/data/n29_sv_insects_results_30/root/ensemble.csv"
)

In [ ]:
/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best_30.txt

In [118]:
sv_insects_category_30 = {}

with open("/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best_30.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_insects_category_30[parts[0]] = general_dict[parts[1]]['class']

In [120]:
def accuracy_from_dicts(y_true: dict[str, str], y_pred: dict[str, str]) -> float:
    common_keys = y_true.keys() & y_pred.keys()
    if not common_keys:
        return 0.0

    correct = sum(y_true[k] == y_pred[k] for k in common_keys)
    return correct / len(common_keys)


acc = accuracy_from_dicts(sv_insects_category_30, name_to_pred_class)
print(acc)

0.811269780007719
